# fit_cpc_model.ipynb





In [1]:
# import hssm
# import hssm.plotting
# import numpy as np
# import pandas as pd
# import arviz as az
# import seaborn as sns
# from matplotlib import pyplot as plt
# import bambi as bmb
# import statsmodels.api as sm
# #from pymer4.models import Lmer
# from scipy import stats
# from scipy.stats import pearsonr
# from matplotlib.patches import Patch
# from matplotlib.colors import to_rgb
# from scipy.interpolate import UnivariateSpline
# import matplotlib.patheffects as pe
# from statsmodels.stats.outliers_influence import variance_inflation_factor
# from numpy.polynomial.legendre import legvander
# import statsmodels.formula.api as smf
# from scipy.stats import norm

import hssm
import numpy as np
import pandas as pd
import arviz as az
from matplotlib import pyplot as plt
from pathlib import Path
#%pip install ydata-profiling
#from ydata_profiling import ProfileReport




Path("model_outputs_07.09_cpc").mkdir(exist_ok=True)

/home/apar355/.conda/envs/hssm_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

#pwd

## Load prepped data

In [3]:
pair_pre = pd.read_csv("hssm_ema_df_pre_07.09.csv")

# make sure subject is string 
pair_pre["subject"] = pair_pre["subject"].astype(str)

# make sure subject is in seconds 
pair_pre["rt"] = pair_pre["rt"].astype(float) / 1000


# make sure response is -1 and 1 / rt is numeric / check range of delta_sb_within_subj_normed / check values of delta_valence / check type and range of affect_z
vars_to_check = ["subject", "response", "rt", "delta_sb_within_subj_normed", "delta_valence", "affect_z"]

summary = pd.DataFrame({
    "dtype": [pair_pre[v].dtype for v in vars_to_check],
    "n_nan": [pair_pre[v].isna().sum() for v in vars_to_check],
    "min": [pair_pre[v].min() if pair_pre[v].dtype != "object" else "—" for v in vars_to_check],
    "max": [pair_pre[v].max() if pair_pre[v].dtype != "object" else "—" for v in vars_to_check],
    "unique": [sorted(pair_pre[v].unique())[:5] if pair_pre[v].nunique() < 10 else f"{pair_pre[v].nunique()} values" for v in vars_to_check],
}, index=vars_to_check)

print(summary)

print(f"PRE:  {len(pair_pre)} trials, {pair_pre['subject'].nunique()} subjects")

print("PRE:", pair_pre.columns.tolist())


                               dtype  n_nan       min       max       unique
subject                       object      0         —         —    25 values
response                       int64      0        -1         1      [-1, 1]
rt                           float64      0     0.221       5.0  3130 values
delta_sb_within_subj_normed  float64      0      -0.5       0.5  1814 values
delta_valence                  int64      0        -1         1   [-1, 0, 1]
affect_z                     float64      0 -3.805755  2.542463   139 values
PRE:  12132 trials, 25 subjects
PRE: ['trial_type', 'trial_index', 'time_elapsed', 'internal_node_id', 'subject', 'session_number', 'session_phase', 'session_day', 'session_slot', 'ema_start_date', 'ema_post_start_date', 'experiment_start', 'browser', 'screen_width', 'screen_height', 'timezone_offset', 'view_history', 'rt', 'stimulus', 'response', 'which_experiment_phase', 'trial_number', 'left_word', 'right_word', 'valence', 'left_base_rating', 'right_base

## FIT MODEL FOR PRE-TRIP DATA 


In [4]:
# MAKE PRE TRIP DATA MODEL 
# in hssm models use delta_sb within subject normalized
cpc_model = hssm.HSSM(
    model="ddm",
    data=pair_pre,
    include=[
        {
            "name": "v",
            "formula": "v ~ 1 + delta_sb_within_subj_normed + delta_valence*affect_z + (1 + delta_sb_within_subj_normed + delta_valence*affect_z | subject)",
            #"formula": "v ~ 1 + r_diff_norm + delta_valence*affect_z + (1 + delta_sb_normed + delta_valence + affect_z | subject)",

        }
    ],
)

cpc_model

Model initialized successfully.


Hierarchical Sequential Sampling Model
Model: ddm

Response variable: rt,response
Likelihood: analytical
Observations: 12132

Parameters:

v:
    Formula: v ~ 1 + delta_sb_within_subj_normed + delta_valence*affect_z + (1 + delta_sb_within_subj_normed + delta_valence*affect_z | subject)
    Priors:
        v_Intercept ~ Normal(mu: 2.0, sigma: 3.0)
        v_delta_sb_within_subj_normed ~ Normal(mu: 0.0, sigma: 0.25)
        v_delta_valence ~ Normal(mu: 0.0, sigma: 0.25)
        v_affect_z ~ Normal(mu: 0.0, sigma: 0.25)
        v_delta_valence:affect_z ~ Normal(mu: 0.0, sigma: 0.25)
        v_1|subject ~ Normal(mu: 0.0, sigma: Weibull(alpha: 1.5, beta: 0.3))
        v_delta_sb_within_subj_normed|subject ~ Normal(mu: Normal(mu: 0.0, sigma: 0.25), sigma: Weibull(alpha: 1.5, beta: 0.3))
        v_delta_valence|subject ~ Normal(mu: Normal(mu: 0.0, sigma: 0.25), sigma: Weibull(alpha: 1.5, beta: 0.3))
        v_affect_z|subject ~ Normal(mu: Normal(mu: 0.0, sigma: 0.25), sigma: Weibull(alpha: 1.

## RUN  MODEL 

In [5]:
ddm_samples_cpc_model = cpc_model.sample(
    sampler = "mcmc",
    tune = 2000, # warmpup samples
    draws = 4000, # 4000
    chains = 4, 
    cores = 4,
    target_accept=0.95,
    #idata_kwargs=dict(log_likelihood=True)  # return log likelihood
    mp_ctx="spawn"
)

ddm_samples_cpc_model.to_netcdf('model_outputs_07.09_cpc/cpc_model.nc')

print("Over")

Using default initvals. 



Initializing NUTS using adapt_diag...


/home/apar355/.conda/envs/hssm_env/lib/python3.11/site-packages/pytensor/link/c/cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [a, t, z, v_Intercept, v_delta_sb_within_subj_normed, v_delta_valence, v_affect_z, v_delta_valence:affect_z, v_1|subject_sigma, v_1|subject_offset, v_delta_sb_within_subj_normed|subject_mu, v_delta_sb_within_subj_normed|subject_sigma, v_delta_sb_within_subj_normed|subject_offset, v_delta_valence|subject_mu, v_delta_valence|subject_sigma, v_delta_valence|subject_offset, v_affect_z|subject_mu, v_affect_z|subject_sigma, v_affect_z|subject_offset, v_delta_valence:affect_z|subject_mu, v_delta_valence:affect_z|subject_sigma, v_delta_valence:affect_z|subject_offset]


/home/apar355/.conda/envs/hssm_env/lib/python3.11/site-packages/rich/live.py:256: UserWarning: install "ipywidgets"
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

## Print model

In [ ]:
ddm_samples_cpc_model

## Graph

In [ ]:
# graph() removed for cluster run — needs dot binary, not needed headless


# PP from hssm resource page

In [ ]:
# PP disabled — regenerate locally from the .nc
